# Quantization of Filter Coefficients

In section 4.4.1 it was shown during the analysis of parameter quantization that the poles are more sensitive as their number increases and the distance between them decreases. For zeros, a similar behavior can be seen. Therefore, the **third canonical form** (exercise 4.4.1) is of great importance for the implementation of higher-order systems.

This exercise shows an example of how **quantization of filter coefficients** influences the system function $H(z)$.
The function `lfxquant()` implements a linear quantizer:

$$
-1 \leq [v]_{Q} \leq 1 - 2^{-l+1} ,
$$

where $l$ is the number of bits and $v$ is the number to be quantized.
To quantize the coefficients of a digital filter, it is necessary to normalize the coefficients.
The function `[aq, n] = lcoefrnd(a, l)` calculates the vector of quantized coefficients `aq` as well as the normalization factor `n`.

---

### a)

For our analysis, we will use a **Cauer low-pass filter** of 7th order with the following specifications:

* $3,\text{dB}$ cut-off frequency: $0.4 \pi$
* Pass-band attenuation: $R_P = 0.1,\text{dB}$
* Stop-band attenuation: $R_S = 40,\text{dB}$

Python’s command (using `scipy.signal.ellip`):

```python
from scipy.signal import ellip
b, a = ellip(7, 0.1, 40, 0.4, btype='low', output='ba')
```

calculates the coefficients.
Plot the transfer function using Python (e.g., `scipy.signal.freqz` and `matplotlib`).

---

### b)

Quantize the filter coefficients with **8 bits**, i.e.:

```python
bq, nb = lcoefrnd(b, 8)
aq, na = lcoefrnd(a, 8)
```

to get the quantized coefficients `bq` and `aq` together with the normalization factors `nb` and `na`.

Plot the resulting transfer functions as well as the version with non-quantized coefficients (exercise 4.4.1.a).

**Hint:** To plot the transfer function of the filter with quantized parameters, the normalization of coefficients has to be inverted:

```python
from scipy.signal import freqz
import matplotlib.pyplot as plt

w, h = freqz(bq, aq, 512)
Hq = nb/na * h
plt.plot(w, 20 * np.log10(abs(Hq)))
plt.show()
```

---

### c)

Applying **8-bit quantization**, there is a high deviation of the transfer function compared to the given tolerance scheme.
The tolerable deviations are:

* $R_D^+ = 0.15,\text{dB}$
* $R_D^- = -0.15,\text{dB}$
* $R_S = -38,\text{dB}$

> $R_D^+$ and $R_D^-$ denote the maximal variations in the pass-band,
> where $R_D^- = R_P$.

Determine the **minimum word length** of the filter coefficients to accomplish this tolerance scheme.

---

There are different structures of digital filters. The **direct form** is implemented in Python using `scipy.signal.lfilter`, which has been used earlier.

To reduce the effects of quantization, the **third canonical form** is mostly used, whereby the system is realized with a number of first- and second-order filters connected in series.

In the following, we analyze the effects of parameter quantization when using the **third canonical form** for the upper Cauer low-pass filter.

---

### d)

Calculate the filter coefficients of the **second-order sections (SOS)** of the Cauer low-pass using `lcascade()` and the resulting transfer function of the system:

$$
H(z) = \prod_{i=1}^{p} H_i(z)
$$

Quantize the filter coefficients of the SOS using `lcoefrnd()` with a word length of $l = 8, 10, 12, \dots$ bits, and calculate the transfer function of the cascade with quantized coefficients.

Determine how many bits are necessary to quantize the filter coefficients of the SOS to fit the tolerance scheme (exercise 4.4.1.a). Compare this length with the minimum length that was necessary for the **direct implementation** of the filter to fit the tolerance scheme (exercise 4.4.1.c).

In [ ]:
# BEGIN SOLUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import ellip, freqz
from helper_functions import lcoefrnd, lcascade

# -------------------------------------------------------------------
# Task a: Design Cauer Lowpass
# -------------------------------------------------------------------

b, a = ellip(7, 0.1, 40, 0.4)
W, H = freqz(b, a, 512)

plt.figure()
plt.plot(W / np.pi, 20 * np.log10(np.abs(H)))
plt.grid()
plt.xlabel('Omega/pi (Nyquist = 1)')
plt.ylabel('|H(exp(j*Omega))| in dB')
plt.title('Cauer Lowpass')


# -------------------------------------------------------------------
# Task b: Quantize filter coefficients
# -------------------------------------------------------------------

b, a = ellip(7, 0.1, 40, 0.4)
W, H = freqz(b, a, 512)

bq, nb = lcoefrnd(b, 8)
aq, na = lcoefrnd(a, 8)
Wq, Hq_temp = freqz(bq, aq, 512)
Hq = (nb / na) * Hq_temp

plt.figure()
plt.plot(W / np.pi, 20 * np.log10(np.abs(H)))
plt.plot(W / np.pi, 20 * np.log10(np.abs(Hq)), '-.')
plt.grid()
plt.title('- Unquantized Coefficients   -. 8-bit Quantized Coefficients')
plt.xlabel('Omega/pi (Nyquist = 1)')
plt.ylabel('|H(exp(j*Omega))| in dB')


# -------------------------------------------------------------------
# Task c: Determine minimum word length
# -------------------------------------------------------------------

b, a = ellip(7, 0.1, 40, 0.4)
W, H = freqz(b, a, 512)
v_length = [8, 12, 16]

for l in v_length:
    bq, nb = lcoefrnd(b, l)
    aq, na = lcoefrnd(a, l)

    _, Hq_temp = freqz(bq, aq, 512)
    Hq = (nb / na) * Hq_temp

    # Passband
    plt.figure()
    plt.plot(W[0:208] / np.pi, 20 * np.log10(np.abs(H[0:208])))
    plt.axis([0, 0.5, -1.4, 0.3])
    plt.grid()
    plt.plot(W[0:208] / np.pi, -0.15 * np.ones(208), 'w')
    plt.plot(W[0:208] / np.pi, +0.15 * np.ones(208), 'w')
    plt.plot(W[0:208] / np.pi, 20 * np.log10(np.abs(Hq[0:208])), '--')
    plt.xlabel('Omega/pi (Nyquist = 1)')
    plt.ylabel('|H(exp(j*Omega))| in dB')
    plt.title(f'Passband: - Unquantized, -- Quantized with {l} Bits')

    # Stopband
    plt.figure()
    plt.plot(W[214:512] / np.pi, 20 * np.log10(np.abs(H[214:512])))
    plt.plot(W[214:512] / np.pi, 20 * np.log10(np.abs(Hq[214:512])), '--')
    plt.plot(W[214:512] / np.pi, -38 * np.ones(512 - 214), 'w')
    plt.grid()
    plt.xlabel('Omega/pi (Nyquist = 1)')
    plt.ylabel('|H(exp(j*Omega))| in dB')
    plt.title(f'Stopband: - Unquantized, -- Quantized with {l} Bits')


# -------------------------------------------------------------------
# Task d: 3rd canonical form quantization
# -------------------------------------------------------------------

b, a = ellip(7, 0.1, 40, 0.4)
B, A = lcascade(b, a)

m, n = B.shape
Hi = np.zeros((512, m), dtype=complex)

for k in range(m):
    _, temp = freqz(B[k, :], A[k, :], 512)
    Hi[:, k] = temp

H3k = np.prod(Hi, axis=1)

# Only for 12 bits, like MATLAB
v_length = [12]

for l in v_length:
    Bq = np.zeros_like(B)
    Aq = np.zeros_like(A)
    nb = np.zeros(m)
    na = np.zeros(m)

    for k in range(m):
        Bq[k, :], nb[k] = lcoefrnd(B[k, :], l)
        Aq[k, :], na[k] = lcoefrnd(A[k, :], l)

    Hq = np.zeros((512, m), dtype=complex)
    for k in range(m):
        _, temp = freqz(Bq[k, :], Aq[k, :], 512)
        Hq[:, k] = (nb[k] / na[k]) * temp

    Hqg = np.prod(Hq, axis=1)

    # Passband
    plt.figure()
    plt.plot(W[0:208] / np.pi, 20 * np.log10(np.abs(H3k[0:208])))
    plt.axis([0, 0.5, -1.4, 0.3])
    plt.grid()
    plt.plot(W[0:208] / np.pi, -0.15 * np.ones(208), 'w')
    plt.plot(W[0:208] / np.pi, +0.15 * np.ones(208), 'w')
    plt.plot(W[0:208] / np.pi, 20 * np.log10(np.abs(Hqg[0:208])), '--')
    plt.xlabel('Omega/pi (Nyquist = 1)')
    plt.ylabel('|H(exp(j*Omega))| in dB')
    plt.title(f'3rd Canonical: Pass.: - Unquantized, -- Quantized with {l} Bits')

    # Stopband
    plt.figure()
    plt.plot(W[214:512] / np.pi, 20 * np.log10(np.abs(H3k[214:512])))
    plt.plot(W[214:512] / np.pi, 20 * np.log10(np.abs(Hqg[214:512])), '--')
    plt.plot(W[214:512] / np.pi, -38 * np.ones(512 - 214), 'w')
    plt.grid()
    plt.xlabel('Omega/pi (Nyquist = 1)')
    plt.ylabel('|H(exp(j*Omega))| in dB')
    plt.title(f'3rd Canonical: Stop.: - Unquantized, -- Quantized with {l} Bits')


plt.show()
# END SOLUTION

## Solution: Quantization of Filter Coefficients

### a)-b)

The figure shows the **amplitude response** of the Cauer low-pass filter.

<img src="./pngs/ldsv4a04_ab.png" width="250">

If the coefficients of both polynomials (numerator and denominator) are quantized according to

$$
-1 \leq [v]_{Q} \leq 1 - 2^{-l+1}
$$

with 8 bits, we obtain the illustrated amplitude response.
A comparison between the two functions shows a clear variation in the transition from **pass-band** to **stop-band**.
The given tolerance scheme is **not fulfilled** by the system with quantized coefficients.

---

### c)

Using the tolerance variations $R_D^+$ and $R_D^-$ given in the exercise, we need at least **16-bit words** for realization in the **direct form** as a single system.

The following figures show the stop- and pass-band of the quantized transfer function in comparison with the non-quantized version:

<img src="./pngs/ldsv4a04_c1.png" width="250">
<img src="./pngs/ldsv4a04_c2.png" width="250">

For a word length of 16 bits, there is a **small variance** before the trailing edge in the pass-band, while in the stop-band, there is no variation.

---

### d)

If we realize the Cauer low-pass (exercise part a) using the **third canonical form** and determine the **minimum quantized word length** needed to fulfill the tolerance scheme, it shows that a **12-bit quantization** is enough.

Compared to the direct implementation (exercise part b), this requires **4 bits less**.
The figures show the amplitude responses in the **stop-** and **pass-band**:

<img src="./pngs/ldsv4a04_d1.png" width="250">
<img src="./pngs/ldsv4a04_d2.png" width="250">